# DO Classifier Test Notebook

Run the DO scene segmenter, crop detected cards, classify each crop with the DO classical full-card classifier (HSV histogram + HOG + Fourier descriptors), and group predictions using the fixed UNO table geometry (`p1` bottom, `p2` right, `p3` top, `p4` left, center in the middle). All logic lives in `src/test_classifier/`; this notebook is a thin orchestrator.

In [ ]:
from pathlib import Path
import sys

candidate_src_dirs = [
    Path.cwd() / "src",
    Path.cwd().parent.parent / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.is_dir():
        parent = src_dir.parent.resolve()
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate do/src directory for imports.")


In [ ]:
from src.test_classifier import (
    TestPipelineConfig,
    initialize_test_pipeline,
    run_single_image_diagnostics,
    run_labeled_benchmark,
)


## Configuration

In [ ]:
# Most-used quick knobs
# Options for IMAGE_SOURCE: "augmented_scene", "train_image", "test_image".
IMAGE_SOURCE = "augmented_scene"
IMAGE_INDEX = 0
IMAGE_ID = None

IMG_SIZE_CLASSIFIER = 128
IMG_SIZE_SEGMENTER = 256
FOURIER_COEFFS = 32
CENTER_CROP_FRACTION = 0.62
EVAL_THRESHOLD = 0.50

EVAL_MAX_IMAGES = None
EVAL_RANDOM_SUBSET = True
EVAL_SEED = 42
WORST_K = 25
TOP_K = 25

CFG = TestPipelineConfig(
    image_source=IMAGE_SOURCE,
    image_index=IMAGE_INDEX,
    image_id=IMAGE_ID,
    img_size_classifier=IMG_SIZE_CLASSIFIER,
    img_size_segmenter=IMG_SIZE_SEGMENTER,
    fourier_coeffs=FOURIER_COEFFS,
    center_crop_fraction=CENTER_CROP_FRACTION,
    eval_threshold=EVAL_THRESHOLD,
)

BENCHMARK_CFG = {
    "eval_max_images": EVAL_MAX_IMAGES,
    "eval_random_subset": EVAL_RANDOM_SUBSET,
    "eval_seed": EVAL_SEED,
    "eval_threshold": CFG.eval_threshold,
    "worst_k": WORST_K,
    "top_k": TOP_K,
}

CFG

## Initialize: load segmenter, classifier, classes, and pick an image

In [ ]:
state = initialize_test_pipeline(CFG)
state.keys()


## Single-image diagnostics: segment, classify, group, and (when available) compare to ground truth

In [ ]:
state = run_single_image_diagnostics(state)
state["summary"]


## Benchmark on the original labeled dataset

> The official `test_images` split has no public labels, so true performance cannot be computed there. This section evaluates the pipeline on labeled images from `train.csv` (optionally on a subset), reports metrics, and visualizes the worst and top performing examples.

In [ ]:
benchmark = run_labeled_benchmark(
    state,
    benchmark_config=BENCHMARK_CFG,
)
{
    k: benchmark[k]
    for k in (
        "center_acc", "p1_f1", "p2_f1", "p3_f1", "p4_f1",
        "macro_f1", "overall", "overall_strict",
        "segmenter_count_quality", "avg_abs_card_count_diff",
        "avg_signed_card_count_diff", "avg_region_abs_card_count_diff",
        "cards_total_true", "cards_total_pred",
    )
}